## Analysis and Lessons Learned

Slurm job `1153220`. These measurements belong to this execution.

### ArtiFact holdout

| model | accuracy | f1_macro | roc_auc |
| --- | --- | --- | --- |
| binary_efficientnet_b4_ns | 0.9376 | 0.9346 | 0.9831 |
| clip_classifier | 0.8131 | 0.8039 | 0.8896 |
| timm_convnextv2_tiny | 0.6158 | 0.3811 | 0.5202 |
| timm_dinov2_vit_base | 0.6254 | 0.6245 | 0.6926 |
| resnet50 | 0.7677 | 0.7562 | 0.8407 |

The highest observed macro F1 is 0.9346, from `binary_efficientnet_b4_ns`.

### CIFAKE external

| model | accuracy | f1_macro | roc_auc |
| --- | --- | --- | --- |
| binary_efficientnet_b4_ns | 0.5000 | 0.3333 | 0.3422 |
| clip_classifier | 0.4988 | 0.3377 | 0.4324 |
| timm_convnextv2_tiny | 0.5000 | 0.3333 | 0.5062 |
| timm_dinov2_vit_base | 0.4621 | 0.4564 | 0.4268 |
| resnet50 | 0.4921 | 0.3786 | 0.4788 |

The highest observed macro F1 is 0.4564, from `timm_dinov2_vit_base`.

### Shoes external

| model | accuracy | f1_macro | roc_auc |
| --- | --- | --- | --- |
| binary_efficientnet_b4_ns | 0.7286 | 0.6700 | 0.8105 |
| clip_classifier | 0.7061 | 0.6732 | 0.7319 |
| timm_convnextv2_tiny | 0.6217 | 0.3834 | 0.4784 |
| timm_dinov2_vit_base | 0.3948 | 0.3233 | 0.4514 |
| resnet50 | 0.5552 | 0.5398 | 0.5571 |

The highest observed macro F1 is 0.6732, from `clip_classifier`.

Macro F1 weights the two classes equally. Compare each model across datasets: an external drop measures transfer on these samples, while a high random holdout score alone does not establish robustness to a new generator or acquisition pipeline. These are single-run comparisons without uncertainty intervals.

### ConvNeXt intervention

| dataset | variant | accuracy | f1_macro | f1_delta |
| --- | --- | --- | --- | --- |
| ArtiFact hold-out | original | 0.6158 | 0.3811 | undefined |
| ArtiFact hold-out | shortcut-resistant | 0.6158 | 0.3811 | 0.0000 |
| CIFAKE external | original | 0.5000 | 0.3333 | undefined |
| CIFAKE external | shortcut-resistant | 0.5000 | 0.3333 | 0.0000 |
| Shoes external | original | 0.6217 | 0.3834 | undefined |
| Shoes external | shortcut-resistant | 0.6217 | 0.3834 | 0.0000 |

The predefined exploratory gate failed. It requires positive macro-F1 changes on both external datasets and an ArtiFact loss of at most 0.03. Because those external sets informed the intervention, a gate pass still requires an untouched confirmation dataset.


## 1. Environment setup
Install optional vision dependencies (`timm`, `kaggle`) so the registry backbones
load without manual package management.

In [1]:
import os
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
repo_root = next((path for path in candidates if (path / "ai_content_detector").is_dir() and (path / "pyproject.toml").is_file()), None)
if repo_root is None:
    raise RuntimeError("Open this notebook from an ai-content-robustness checkout after installing its dependencies")
os.chdir(repo_root)
print(f"Using checkout: {repo_root}")


Using checkout: /lustre/fswork/projects/rech/mbm/ual37ci/bench_research_ml_project/split-repos/ai-content-robustness


In [2]:
print(f"Working directory: {Path.cwd()}")


Working directory: /lustre/fswork/projects/rech/mbm/ual37ci/bench_research_ml_project/split-repos/ai-content-robustness


## 2. Imports and deterministic utilities
Everything important (models, metrics, benchmarking) is imported from the shared
library so we avoid redefining models or augmentations inside the notebook.

In [3]:
import os
from pathlib import Path
from typing import Iterable
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from ml_pipeline.pipelines_torch.benchmark import BenchmarkRunner
from ml_pipeline.pipelines_torch.vision_models import MODEL_REGISTRY as VISION_MODELS
from ml_pipeline.pipelines_torch.base import SimplePredictor
from ml_pipeline.utils.metrics import METRIC_REGISTRY
from ml_pipeline.utils.utils import load_model_by_name, RESULTS_DIR
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 3. Download and prepare ArtiFact
The ArtiFact dataset contains ~2.5M images (965K real + 1.5M fake) from 25 different
generators at 200×200 resolution. This provides much more generator diversity than CIFAKE
(single generator). We use stratified 15% sampling to keep training tractable.

In [4]:
from pathlib import Path

from ml_pipeline.utils.kaggle_utils import ensure_kaggle_dataset

KAGGLE_ARTIFACT = "awsaf49/artifact-dataset"
DATA_DIR = ensure_kaggle_dataset(
    dataset_slug=KAGGLE_ARTIFACT,
    local_dir=Path("data/artifact-real-fake"),
    description="ArtiFact dataset",
    kaggle_subdir="artifact-dataset",
)

if DATA_DIR.exists():
    print(f"✅ ArtiFact data available at {DATA_DIR}")
else:
    print(f"⚠️ ArtiFact dataset missing at {DATA_DIR}")

ℹ️ ArtiFact dataset already present at data/artifact-real-fake
✅ ArtiFact data available at data/artifact-real-fake


In [5]:
IMG_SIZE = 200
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

from torch.utils.data import Dataset
from torchvision.datasets.folder import IMG_EXTENSIONS, default_loader

IMAGE_EXTENSIONS = tuple(ext.lower() for ext in IMG_EXTENSIONS)


def _norm_name(value: str) -> str:
    return "".join(ch for ch in str(value).lower() if ch.isalnum())


def _has_image_file(path: Path) -> bool:
    path = Path(path)
    return any(
        child.is_file() and child.suffix.lower() in IMAGE_EXTENSIONS
        for child in path.rglob("*")
    )


def is_imagefolder_dir(path: Path) -> bool:
    path = Path(path)
    if not path.is_dir():
        return False
    class_dirs = [p for p in path.iterdir() if p.is_dir()]
    # torchvision ImageFolder accepts recursive files under each class directory.
    return sum(_has_image_file(class_dir) for class_dir in class_dirs) >= 2


def find_imagefolder_split(base_dir: Path) -> Path:
    base_dir = Path(base_dir)
    if is_imagefolder_dir(base_dir):
        return base_dir
    preferred = ("train", "test", "validation", "val", "eval")
    for name in preferred:
        for variant in {name, name.upper(), name.capitalize()}:
            candidate = base_dir / variant
            if is_imagefolder_dir(candidate):
                return candidate
    for candidate in sorted(p for p in base_dir.rglob("*") if p.is_dir()):
        if is_imagefolder_dir(candidate):
            return candidate
    raise ValueError(f"Could not locate an ImageFolder split inside {base_dir}")


REAL_CLASS_MARKERS = {
    "real", "reals", "true", "truth", "factual", "authentic", "natural",
}
FAKE_CLASS_MARKERS = {
    "fake", "fakes", "false", "artificial", "synthetic", "generated", "ai", "aigenerated",
}

# Common ArtiFact real source folders and fake generator folders. Folder names are
# normalized before matching, so e.g. "stable-diffusion" and "stable_diffusion" match.
REAL_SOURCE_MARKERS = {
    "afhq", "celebahq", "celeba", "coco", "ffhq", "imagenet", "landscape",
    "lsun", "metfaces", "places", "places365",
}
FAKE_SOURCE_MARKERS = {
    "biggan", "cyclegan", "dalle", "dalle2", "ddpm", "denoisingdiffusiongan",
    "gaugan", "generativeinpainting", "glide", "lama", "latentdiffusion",
    "mat", "palette", "projectedgan", "progan", "sfhq", "stablediffusion",
    "stargan", "stylegan", "stylegan2", "stylegan3", "tamingtransformer",
    "vqdiffusion", "vqgan", "wukong",
}


def _binary_label_from_path(image_path: Path, root: Path):
    parts = [_norm_name(part) for part in image_path.relative_to(root).parts[:-1]]
    for part in parts:
        if part in REAL_CLASS_MARKERS:
            return 1
        if part in FAKE_CLASS_MARKERS:
            return 0
    for part in parts:
        if part in REAL_SOURCE_MARKERS:
            return 1
        if part in FAKE_SOURCE_MARKERS:
            return 0
    return None


class BinaryImagePathDataset(Dataset):
    classes = ["fake", "real"]
    class_to_idx = {"fake": 0, "real": 1}

    def __init__(self, samples, transform=None):
        self.samples = [(Path(path), int(target)) for path, target in samples]
        self.targets = [target for _, target in self.samples]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, target = self.samples[index]
        image = default_loader(path)
        if self.transform is not None:
            image = self.transform(image)
        return image, target


def _is_binary_imagefolder(classes) -> bool:
    normalized = {_norm_name(name) for name in classes}
    has_real = bool(normalized & (REAL_CLASS_MARKERS | REAL_SOURCE_MARKERS))
    has_fake = bool(normalized & (FAKE_CLASS_MARKERS | FAKE_SOURCE_MARKERS))
    return len(classes) == 2 and has_real and has_fake


def build_binary_image_dataset(base_dir: Path, transform=None) -> BinaryImagePathDataset:
    base_dir = Path(base_dir)
    image_files = sorted(
        p for p in base_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )
    samples = []
    skipped = 0
    for image_path in image_files:
        label = _binary_label_from_path(image_path, base_dir)
        if label is None:
            skipped += 1
            continue
        samples.append((image_path, label))

    labels = {target for _, target in samples}
    if labels != {0, 1}:
        preview_dirs = [str(p.relative_to(base_dir)) for p in sorted(base_dir.iterdir())[:20]] if base_dir.exists() else []
        raise ValueError(
            f"Could not infer both fake and real ArtiFact labels inside {base_dir}. "
            f"Found {len(samples)} labelled images across labels {sorted(labels)}; "
            f"skipped {skipped}. Top-level entries: {preview_dirs}"
        )
    if skipped:
        print(f"Skipped {skipped:,} images whose ArtiFact label could not be inferred from the path")
    return BinaryImagePathDataset(samples, transform=transform)


def load_binary_image_dataset(base_dir: Path, transform=None):
    base_dir = Path(base_dir)
    try:
        split = find_imagefolder_split(base_dir)
        imagefolder = datasets.ImageFolder(split, transform=transform)
        if _is_binary_imagefolder(imagefolder.classes):
            return imagefolder, split
        print(
            f"Found ImageFolder-like directory at {split}, but classes are {imagefolder.classes}; "
            "building binary fake/real labels from ArtiFact path names instead."
        )
    except ValueError:
        pass

    dataset = build_binary_image_dataset(base_dir, transform=transform)
    return dataset, base_dir


train_ds, train_split = load_binary_image_dataset(DATA_DIR, transform=transform)
print(f"Using ArtiFact data from: {train_split}")

class_names = train_ds.classes
num_classes = len(class_names)
print(f"Classes: {class_names}")
print(f"Total samples: {len(train_ds)}")


Found ImageFolder-like directory at data/artifact-real-fake, but classes are ['afhq', 'big_gan', 'celebahq', 'cips', 'coco', 'cycle_gan', 'ddpm', 'denoising_diffusion_gan', 'diffusion_gan', 'face_synthetics', 'ffhq', 'gansformer', 'gau_gan', 'generative_inpainting', 'glide', 'imagenet', 'lama', 'landscape', 'latent_diffusion', 'lsun', 'mat', 'metfaces', 'palette', 'pro_gan', 'projected_gan', 'sfhq', 'stable_diffusion', 'star_gan', 'stylegan1', 'stylegan2', 'stylegan3', 'taming_transformer', 'vq_diffusion']; building binary fake/real labels from ArtiFact path names instead.


Skipped 56,707 images whose ArtiFact label could not be inferred from the path


Using ArtiFact data from: data/artifact-real-fake
Classes: ['fake', 'real']
Total samples: 2440031


In [6]:
def dataset_to_numpy(dataset: datasets.ImageFolder):
    tensors = [img for img, _ in dataset]
    X = torch.stack(tensors).numpy()
    y = np.array(dataset.targets, dtype=np.int64)
    return X.astype(np.float32), y

SAMPLE_FRACTION = 0.15
print(f"Original ArtiFact size: {len(train_ds)} images")

selected_indices, _, selected_labels, _ = train_test_split(
    np.arange(len(train_ds)), np.asarray(train_ds.targets),
    train_size=SAMPLE_FRACTION,
    random_state=SEED,
    stratify=np.asarray(train_ds.targets),
)

train_indices, test_indices, y_all, y_test_artifact = train_test_split(
    selected_indices, selected_labels, test_size=0.2, random_state=SEED, stratify=selected_labels,
)


def materialize_images(dataset, indices):
    X = np.empty((len(indices), 3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    for output_index, source_index in enumerate(indices):
        X[output_index] = dataset[int(source_index)][0].numpy()
    return X


X_all = materialize_images(train_ds, train_indices)
X_test_artifact = materialize_images(train_ds, test_indices)
print(f"Training data shape: {X_all.shape}")
print(f"Held-out ArtiFact test shape: {X_test_artifact.shape}")
print(f"Class distribution: {np.bincount(y_all)}")

Original ArtiFact size: 2440031 images


Training data shape: (292803, 3, 200, 200)
Held-out ArtiFact test shape: (73201, 3, 200, 200)
Class distribution: [180317 112486]


## 4. Configure `BenchmarkRunner`
We select a focused subset spanning complementary pretraining strategies:
- **binary_efficientnet_b4_ns** — strong supervised CNN baseline
- **clip_classifier** — vision-language pretrained representation
- **timm_convnextv2_tiny** — strong modern ConvNet backbone
- **timm_dinov2_vit_base** — DINOv2 self-supervised ViT
- **resnet50** — standard baseline for comparison

In [7]:
SELECTED_MODELS = [
    "binary_efficientnet_b4_ns",
    "clip_classifier",
    "timm_convnextv2_tiny",
    "timm_dinov2_vit_base",
    "resnet50",
]

def model_params(name):
    params = {"num_classes": num_classes}
    if name.startswith("timm_"):
        # dinov2 uses patch size 14, so the input side must be divisible by 14
        params["img_size"] = 196 if "dinov2" in name else IMG_SIZE
    return params

model_configs = []
epochs = {}
for name in SELECTED_MODELS:
    if name not in VISION_MODELS:
        raise KeyError(f"{name} is not registered in ml_pipeline.pipelines_torch.vision_models")
    model_configs.append({
        "name": name,
        "class": VISION_MODELS[name],
        "params": model_params(name),
    })
    if name == "timm_convnextv2_tiny":
        model_configs[-1]["use_mixed_precision"] = False
    epochs[name] = 7


runner = BenchmarkRunner(
    model_configs=model_configs,
    augmentations=[None],
    task_type="classification",
    device=DEVICE,
    epochs=epochs,
    batch_size=32,
    early_stopping=None,
    use_class_weights=True,
    use_kfold=False,
    learning_rate=3e-4,
    use_wandb=True,
    path_start="bench_imai_artifact",
    random_state=SEED,
)
results_df = runner.run(X_all, y_all)
results_df

Models:   0%|          | 0/5 [00:00<?, ?it/s]

Augmentations:   0%|          | 0/1 [00:00<?, ?it/s]

Augmentations: 100%|██████████| 1/1 [00:00<00:00,  3.16it/s]

Models:  20%|██        | 1/5 [00:00<00:01,  3.13it/s]

Augmentations:   0%|          | 0/1 [00:00<?, ?it/s]

...

wandb: Find logs at: ./wandb/offline-run-20261018_054951-g635d5cz/logs


Augmentations: 100%|██████████| 1/1 [11:34<00:00, 694.09s/it]

Models: 100%|██████████| 5/5 [1:48:03<00:00, 1309.38s/it]

Models: 100%|██████████| 5/5 [1:48:03<00:00, 1296.73s/it]

Model state dict saved to results/bench_imai_artifact/02f4bc26f454.pt


In [8]:
import os
import tarfile

from ml_pipeline.utils.kaggle_utils import _running_on_kaggle

if _running_on_kaggle():
    from IPython.display import FileLink
    os.chdir('/kaggle/working/results')
    with tarfile.open('/kaggle/working/all_files.tar.gz', 'w:gz') as tar:
        for root, dirs, files in os.walk('.'):
            for file in files:
                tar.add(os.path.join(root, file))
    os.chdir('/kaggle/working')
    FileLink('all_files.tar.gz')
else:
    print("Not on Kaggle; checkpoints stay under results/")


Not on Kaggle; checkpoints stay under results/


## 6. Evaluate the ArtiFact test split
Re-use the same helper to score a held-out ArtiFact split.

In [9]:
def evaluate_saved_models(
    model_names: Iterable[str], X: np.ndarray, y: np.ndarray,
    path_start: str = "bench_imai_artifact",
) -> pd.DataFrame:
    metric_map = {
        "accuracy": "accuracy",
        "f1_macro": "f1",
        "precision_macro": "precision",
        "recall_macro": "recall",
        "roc_auc": "roc_auc",
        "pr_auc": "pr_auc",
    }
    records = []
    for name in model_names:
        try:
            model = load_model_by_name(VISION_MODELS[name], name, model_params(name), path_start=path_start)
        except FileNotFoundError:
            print(f"⚠️ Skipping {name}: checkpoint not found")
            continue
        print(name)
        predictor = SimplePredictor(model, task_type="classification", device=DEVICE, batch_size=32)
        probs = predictor.predict_proba(X)
        scores = {
            label: float(METRIC_REGISTRY[key](y, probs))
            for label, key in metric_map.items()
        }
        records.append({
            "model": name,
            **scores,
        })
    return pd.DataFrame.from_records(records)

In [10]:
def find_optional_artifact_test_dataset(base_dir: Path):
    test_candidates = [
        base_dir / "test",
        base_dir / "Test",
        base_dir / "val",
        base_dir / "validation",
        base_dir / "eval",
    ]
    for candidate in test_candidates:
        if candidate.exists() and any(candidate.iterdir()):
            try:
                test_dataset, source = load_binary_image_dataset(candidate, transform=transform)
                if len(test_dataset) > 0:
                    return test_dataset, source
            except Exception as exc:
                print(f"Skipping {candidate}: {exc}")
    return None, None


test_ds, artifact_test_dir = find_optional_artifact_test_dataset(DATA_DIR)

if test_ds is not None:
    print(f"Found ArtiFact test split at: {artifact_test_dir}")
    X_test_artifact, y_test_artifact = dataset_to_numpy(test_ds)
else:
    print("No dedicated test split found; using the 20% hold-out carved out before training")

print(f"ArtiFact test shape: {X_test_artifact.shape}")


No dedicated test split found; using the 20% hold-out carved out before training
ArtiFact test shape: (73201, 3, 200, 200)


In [11]:
artifact_test_metrics = evaluate_saved_models(SELECTED_MODELS, X_test_artifact, y_test_artifact)
artifact_test_metrics.sort_values("accuracy", ascending=False)

Loading model from: results/bench_imai_artifact/cc1c1eb38018.pt


/lustre/fswork/projects/rech/mbm/ual37ci/bench_research_ml_project/split-repos/ai-content-robustness/.venv/lib/python3.12/site-packages/timm/models/_factory.py:117: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  model = create_fn(


✓ PyTorch model loaded from results/bench_imai_artifact/cc1c1eb38018.pt
binary_efficientnet_b4_ns


Loading model from: results/bench_imai_artifact/5a76d14662ce.pt


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading model from: results/bench_imai_artifact/056c25388587.pt


✓ PyTorch model loaded from results/bench_imai_artifact/056c25388587.pt
timm_dinov2_vit_base


Loading model from: results/bench_imai_artifact/02f4bc26f454.pt


✓ PyTorch model loaded from results/bench_imai_artifact/02f4bc26f454.pt
resnet50


,model,accuracy,f1_macro,precision_macro,recall_macro,roc_auc,pr_auc
0,binary_efficientnet_b4_ns,0.937569,0.934574,0.931451,0.938369,0.983099,0.969940
1,clip_classifier,0.813131,0.803895,0.802059,0.806121,0.889589,0.824147
4,resnet50,0.767722,0.756229,0.754708,0.758109,0.840745,0.747826
3,timm_dinov2_vit_base,0.625401,0.624524,0.643660,0.649041,0.692587,0.528848
2,timm_convnextv2_tiny,0.615839,0.381126,0.307919,0.500000,0.520192,0.393879


## 7. CIFAKE cross-dataset generalisation check
Test whether ArtiFact-trained models transfer to the lower-resolution CIFAKE
test set (32×32, single generator).

In [12]:
from ml_pipeline.utils.kaggle_utils import ensure_kaggle_dataset

KAGGLE_CIFAKE = "birdy654/cifake-real-and-ai-generated-synthetic-images"
CIFAKE_DIR = ensure_kaggle_dataset(
    dataset_slug=KAGGLE_CIFAKE,
    local_dir=Path("data/cifake"),
    description="CIFAKE dataset",
    kaggle_subdir="cifake-real-and-ai-generated-synthetic-images",
)

CIFAKE_SIZE = 32
cifake_transform = transforms.Compose([
    transforms.Resize((CIFAKE_SIZE, CIFAKE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

cifake_test_dir = CIFAKE_DIR / "test"
if not cifake_test_dir.exists():
    raise FileNotFoundError("Expected CIFAKE to expose a test/ split")

cifake_test_ds = datasets.ImageFolder(cifake_test_dir, transform=cifake_transform)
X_test_cifake, y_test_cifake = dataset_to_numpy(cifake_test_ds)
print(f"CIFAKE test shape: {X_test_cifake.shape}")

cifake_test_metrics = evaluate_saved_models(SELECTED_MODELS, X_test_cifake, y_test_cifake)
cifake_test_metrics.sort_values("accuracy", ascending=False)

ℹ️ CIFAKE dataset already present at data/cifake


CIFAKE test shape: (20000, 3, 32, 32)
Loading model from: results/bench_imai_artifact/cc1c1eb38018.pt


/lustre/fswork/projects/rech/mbm/ual37ci/bench_research_ml_project/split-repos/ai-content-robustness/.venv/lib/python3.12/site-packages/timm/models/_factory.py:117: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  model = create_fn(


✓ PyTorch model loaded from results/bench_imai_artifact/cc1c1eb38018.pt
binary_efficientnet_b4_ns


Loading model from: results/bench_imai_artifact/5a76d14662ce.pt


...

Loading model from: results/bench_imai_artifact/056c25388587.pt


✓ PyTorch model loaded from results/bench_imai_artifact/056c25388587.pt
timm_dinov2_vit_base


Loading model from: results/bench_imai_artifact/02f4bc26f454.pt


✓ PyTorch model loaded from results/bench_imai_artifact/02f4bc26f454.pt
resnet50


,model,accuracy,f1_macro,precision_macro,recall_macro,roc_auc,pr_auc
0,binary_efficientnet_b4_ns,0.50000,0.333333,0.250000,0.50000,0.342193,0.396509
2,timm_convnextv2_tiny,0.50000,0.333333,0.250000,0.50000,0.506232,0.495325
1,clip_classifier,0.49880,0.337697,0.455580,0.49880,0.432429,0.447841
4,resnet50,0.49210,0.378586,0.470667,0.49210,0.478755,0.480142
3,timm_dinov2_vit_base,0.46205,0.456400,0.460404,0.46205,0.426828,0.438240


## 8. Shoes dataset generalisation check
The Shoes dataset tests a different distribution shift: high-resolution images
from a single generator (Midjourney).

In [13]:
from ml_pipeline.utils.kaggle_utils import ensure_kaggle_dataset

KAGGLE_SHOES = "sunnykakar/shoes-dataset-real-and-ai-generated-images"
SHOES_DIR = ensure_kaggle_dataset(
    dataset_slug=KAGGLE_SHOES,
    local_dir=Path("data/shoes-ai-vs-real"),
    description="SunnyKakar Shoes (AI vs Real)",
    kaggle_subdir="shoes-dataset-real-and-ai-generated-images",
)

shoes_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

shoes_split = find_imagefolder_split(SHOES_DIR)
print(f"Using {shoes_split} for evaluation")

shoes_ds = datasets.ImageFolder(shoes_split, transform=shoes_transform)
X_test_shoes, y_test_shoes = dataset_to_numpy(shoes_ds)
print(f"Shoes test shape: {X_test_shoes.shape}")

shoes_metrics = evaluate_saved_models(SELECTED_MODELS, X_test_shoes, y_test_shoes)
shoes_metrics.sort_values("accuracy", ascending=False)

ℹ️ SunnyKakar Shoes (AI vs Real) already present at data/shoes-ai-vs-real
Using data/shoes-ai-vs-real for evaluation


Shoes test shape: (2181, 3, 200, 200)
Loading model from: results/bench_imai_artifact/cc1c1eb38018.pt


/lustre/fswork/projects/rech/mbm/ual37ci/bench_research_ml_project/split-repos/ai-content-robustness/.venv/lib/python3.12/site-packages/timm/models/_factory.py:117: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  model = create_fn(


✓ PyTorch model loaded from results/bench_imai_artifact/cc1c1eb38018.pt
binary_efficientnet_b4_ns


Loading model from: results/bench_imai_artifact/5a76d14662ce.pt


...

Loading model from: results/bench_imai_artifact/056c25388587.pt


✓ PyTorch model loaded from results/bench_imai_artifact/056c25388587.pt
timm_dinov2_vit_base


Loading model from: results/bench_imai_artifact/02f4bc26f454.pt


✓ PyTorch model loaded from results/bench_imai_artifact/02f4bc26f454.pt
resnet50


,model,accuracy,f1_macro,precision_macro,recall_macro,roc_auc,pr_auc
0,binary_efficientnet_b4_ns,0.728565,0.669968,0.742814,0.665420,0.810455,0.716653
1,clip_classifier,0.706098,0.673199,0.687108,0.668474,0.731934,0.636871
2,timm_convnextv2_tiny,0.621733,0.383376,0.310867,0.500000,0.478417,0.375004
4,resnet50,0.555250,0.539849,0.540961,0.542890,0.557119,0.457427
3,timm_dinov2_vit_base,0.394773,0.323257,0.517659,0.503781,0.451381,0.349485


Historical motivation: the numerical values below belong to an earlier execution. Current measurements and the intervention decision appear in the final analysis.

# 9. Failure diagnosis and improvement experiment

The original experiment is intentionally retained as a negative result. ConvNeXt reaches
**99.9% accuracy / 99.8% macro F1** on the random ArtiFact hold-out, but only **50.0% /
33.3%** on CIFAKE and **63.2% / 60.0%** on Shoes. Most other models collapse toward
single-class predictions on CIFAKE. High in-domain accuracy therefore does not demonstrate
that the detectors learned generator-independent forensic evidence.

The gap is consistent with shortcut learning from source-specific resolution, compression,
colour pipeline, and generator artefacts. The random ArtiFact split preserves those shortcuts
on both sides of the split. CIFAKE then changes resolution from 200 to 32 pixels and both
external datasets change the acquisition and encoding pipelines.

The intervention below retrains only ConvNeXt V2 Tiny on a class-balanced sample with random
downsample/upsample, JPEG recompression, blur, colour perturbation, and horizontal flipping.
It uses ArtiFact training images only. Because the external results were inspected before this
intervention was designed, this is exploratory and still requires an untouched confirmation set.

In [14]:
from io import BytesIO
from PIL import Image, ImageEnhance, ImageFilter

ROBUST_SAMPLES_PER_CLASS = int(np.bincount(y_all).min())
ROBUST_CLEAN_PROBABILITY = 0.15
ROBUST_RESOLUTIONS = (32, 48, 64, 96, 128, 160, 200)
ROBUST_JPEG_QUALITIES = (30, 40, 50, 60, 70, 80, 90, 100)
_mean = np.asarray(IMAGENET_MEAN, dtype=np.float32)[:, None, None]
_std = np.asarray(IMAGENET_STD, dtype=np.float32)[:, None, None]


def shortcut_resistant_view(image_chw: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    if rng.random() < ROBUST_CLEAN_PROBABILITY:
        return image_chw.copy()
    rgb = np.clip(image_chw * _std + _mean, 0.0, 1.0)
    image = Image.fromarray((rgb.transpose(1, 2, 0) * 255).round().astype(np.uint8))
    if rng.random() < 0.85:
        side = int(rng.choice(ROBUST_RESOLUTIONS))
        image = image.resize((side, side), Image.Resampling.BICUBIC)
        image = image.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
    if rng.random() < 0.80:
        buffer = BytesIO()
        image.save(buffer, format="JPEG", quality=int(rng.choice(ROBUST_JPEG_QUALITIES)))
        buffer.seek(0)
        image = Image.open(buffer).convert("RGB").copy()
    if rng.random() < 0.35:
        image = image.filter(ImageFilter.GaussianBlur(radius=float(rng.uniform(0.1, 1.5))))
    if rng.random() < 0.50:
        image = ImageEnhance.Brightness(image).enhance(float(rng.uniform(0.8, 1.2)))
        image = ImageEnhance.Contrast(image).enhance(float(rng.uniform(0.8, 1.2)))
        image = ImageEnhance.Color(image).enhance(float(rng.uniform(0.8, 1.2)))
    if rng.random() < 0.50:
        image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    transformed = np.asarray(image, dtype=np.float32).transpose(2, 0, 1) / 255.0
    return ((transformed - _mean) / _std).astype(np.float32)


robust_rng = np.random.default_rng(SEED)
robust_source_indices, robust_labels = [], []
for label in np.unique(y_all):
    candidates = np.flatnonzero(y_all == label)
    selected = robust_rng.choice(
        candidates, size=ROBUST_SAMPLES_PER_CLASS,
        replace=False,
    )
    robust_source_indices.extend(selected.tolist())
    robust_labels.extend([int(label)] * ROBUST_SAMPLES_PER_CLASS)
order = robust_rng.permutation(len(robust_source_indices))
robust_source_indices = np.asarray(robust_source_indices)[order]
y_robust = np.asarray(robust_labels, dtype=np.int64)[order]
X_robust = np.empty((len(order), 3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
for output_idx, source_idx in enumerate(robust_source_indices):
    X_robust[output_idx] = shortcut_resistant_view(X_all[source_idx], robust_rng)
del X_all
print(f"Robust training shape: {X_robust.shape}")
print(f"Class distribution: {np.bincount(y_robust)}")
print(f"Memory: {X_robust.nbytes / 1024**3:.2f} GiB")

Robust training shape: (224972, 3, 200, 200)
Class distribution: [112486 112486]
Memory: 100.57 GiB


In [15]:
ROBUST_MODEL = "timm_convnextv2_tiny"
ROBUST_PATH = "bench_imai_artifact_robust"
robust_runner = BenchmarkRunner(
    model_configs=[{"name": ROBUST_MODEL, "class": VISION_MODELS[ROBUST_MODEL],
                    "params": model_params(ROBUST_MODEL), "use_mixed_precision": False}],
    augmentations=[None], task_type="classification", device=DEVICE,
    epochs={ROBUST_MODEL: 10}, batch_size=32, early_stopping=None,
    use_class_weights=False, use_kfold=False, learning_rate=3e-4,
    use_wandb=True, path_start=ROBUST_PATH, random_state=SEED,
)
robust_training_results = robust_runner.run(X_robust, y_robust)
robust_training_results

Models:   0%|          | 0/1 [00:00<?, ?it/s]

Augmentations:   0%|          | 0/1 [00:00<?, ?it/s]

Dropping unsupported 'img_size' argument for model convnextv2_tiny.fcmae_ft_in1k


wandb: Tracking run with wandb version 0.28.1


wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /lustre/fswork/projects/rech/mbm/ual37ci/bench_research_ml_project/split-repos/ai-content-robustness/wandb/offline-run-20261018_061803-iof97ois
wandb: View this run in the terminal with `wandb leet`


...

wandb: You can sync this run to the cloud by running:
wandb: wandb sync /lustre/fswork/projects/rech/mbm/ual37ci/bench_research_ml_project/split-repos/ai-content-robustness/wandb/offline-run-20261018_061803-iof97ois


wandb: Find logs at: ./wandb/offline-run-20261018_061803-iof97ois/logs


Augmentations: 100%|██████████| 1/1 [1:06:44<00:00, 4004.21s/it]

Models: 100%|██████████| 1/1 [1:06:44<00:00, 4004.25s/it]

Models: 100%|██████████| 1/1 [1:06:44<00:00, 4004.25s/it]

In [16]:
robust_eval_sets = {
    "ArtiFact hold-out": (X_test_artifact, y_test_artifact, artifact_test_metrics),
    "CIFAKE external": (X_test_cifake, y_test_cifake, cifake_test_metrics),
    "Shoes external": (X_test_shoes, y_test_shoes, shoes_metrics),
}
comparison_rows = []
for dataset_name, (X_eval, y_eval, baseline_table) in robust_eval_sets.items():
    baseline_row = baseline_table.query("model == @ROBUST_MODEL").iloc[0]
    robust_row = evaluate_saved_models(
        [ROBUST_MODEL], X_eval, y_eval, path_start=ROBUST_PATH
    ).iloc[0]
    for variant, row in (("original", baseline_row), ("shortcut-resistant", robust_row)):
        comparison_rows.append({
            "dataset": dataset_name, "variant": variant,
            "accuracy": row["accuracy"], "f1_macro": row["f1_macro"],
            "roc_auc": row["roc_auc"],
        })
robust_comparison = pd.DataFrame(comparison_rows)
robust_comparison["f1_delta"] = robust_comparison.groupby("dataset")["f1_macro"].diff()
robust_comparison

Loading model from: results/bench_imai_artifact_robust/fd733b707d36.pt
Dropping unsupported 'img_size' argument for model convnextv2_tiny.fcmae_ft_in1k


✓ PyTorch model loaded from results/bench_imai_artifact_robust/fd733b707d36.pt
timm_convnextv2_tiny


Loading model from: results/bench_imai_artifact_robust/fd733b707d36.pt
Dropping unsupported 'img_size' argument for model convnextv2_tiny.fcmae_ft_in1k


✓ PyTorch model loaded from results/bench_imai_artifact_robust/fd733b707d36.pt
timm_convnextv2_tiny


Loading model from: results/bench_imai_artifact_robust/fd733b707d36.pt
Dropping unsupported 'img_size' argument for model convnextv2_tiny.fcmae_ft_in1k


✓ PyTorch model loaded from results/bench_imai_artifact_robust/fd733b707d36.pt
timm_convnextv2_tiny


,dataset,variant,accuracy,f1_macro,roc_auc,f1_delta
0,ArtiFact hold-out,original,0.615839,0.381126,0.520192,NaN
1,ArtiFact hold-out,shortcut-resistant,0.615839,0.381126,0.517093,0.0
2,CIFAKE external,original,0.500000,0.333333,0.506232,NaN
3,CIFAKE external,shortcut-resistant,0.500000,0.333333,0.566972,0.0
4,Shoes external,original,0.621733,0.383376,0.478417,NaN
5,Shoes external,shortcut-resistant,0.621733,0.383376,0.911210,0.0


In [17]:
external_deltas = robust_comparison.query(
    "variant == 'shortcut-resistant' and dataset != 'ArtiFact hold-out'"
)["f1_delta"]
artifact_delta = robust_comparison.query(
    "variant == 'shortcut-resistant' and dataset == 'ArtiFact hold-out'"
)["f1_delta"].iloc[0]
passes_exploratory_gate = bool((external_deltas > 0).all() and artifact_delta >= -0.03)
print(f"Improves macro F1 on both external datasets: {(external_deltas > 0).all()}")
print(f"ArtiFact macro-F1 change: {artifact_delta:+.4f}")
print(f"Passes exploratory gate: {passes_exploratory_gate}")
if passes_exploratory_gate:
    print("Candidate retained; validate once on an untouched external dataset.")
else:
    print("Intervention rejected; keep the original failure as the conclusion.")

Improves macro F1 on both external datasets: False
ArtiFact macro-F1 change: +0.0000
Passes exploratory gate: False
Intervention rejected; keep the original failure as the conclusion.


## Analysis and Lessons Learned

Slurm job `1153220`. These measurements belong to this execution.

### ArtiFact holdout

| model | accuracy | f1_macro | roc_auc |
| --- | --- | --- | --- |
| binary_efficientnet_b4_ns | 0.9376 | 0.9346 | 0.9831 |
| clip_classifier | 0.8131 | 0.8039 | 0.8896 |
| timm_convnextv2_tiny | 0.6158 | 0.3811 | 0.5202 |
| timm_dinov2_vit_base | 0.6254 | 0.6245 | 0.6926 |
| resnet50 | 0.7677 | 0.7562 | 0.8407 |

The highest observed macro F1 is 0.9346, from `binary_efficientnet_b4_ns`.

### CIFAKE external

| model | accuracy | f1_macro | roc_auc |
| --- | --- | --- | --- |
| binary_efficientnet_b4_ns | 0.5000 | 0.3333 | 0.3422 |
| clip_classifier | 0.4988 | 0.3377 | 0.4324 |
| timm_convnextv2_tiny | 0.5000 | 0.3333 | 0.5062 |
| timm_dinov2_vit_base | 0.4621 | 0.4564 | 0.4268 |
| resnet50 | 0.4921 | 0.3786 | 0.4788 |

The highest observed macro F1 is 0.4564, from `timm_dinov2_vit_base`.

### Shoes external

| model | accuracy | f1_macro | roc_auc |
| --- | --- | --- | --- |
| binary_efficientnet_b4_ns | 0.7286 | 0.6700 | 0.8105 |
| clip_classifier | 0.7061 | 0.6732 | 0.7319 |
| timm_convnextv2_tiny | 0.6217 | 0.3834 | 0.4784 |
| timm_dinov2_vit_base | 0.3948 | 0.3233 | 0.4514 |
| resnet50 | 0.5552 | 0.5398 | 0.5571 |

The highest observed macro F1 is 0.6732, from `clip_classifier`.

Macro F1 weights the two classes equally. Compare each model across datasets: an external drop measures transfer on these samples, while a high random holdout score alone does not establish robustness to a new generator or acquisition pipeline. These are single-run comparisons without uncertainty intervals.

### ConvNeXt intervention

| dataset | variant | accuracy | f1_macro | f1_delta |
| --- | --- | --- | --- | --- |
| ArtiFact hold-out | original | 0.6158 | 0.3811 | undefined |
| ArtiFact hold-out | shortcut-resistant | 0.6158 | 0.3811 | 0.0000 |
| CIFAKE external | original | 0.5000 | 0.3333 | undefined |
| CIFAKE external | shortcut-resistant | 0.5000 | 0.3333 | 0.0000 |
| Shoes external | original | 0.6217 | 0.3834 | undefined |
| Shoes external | shortcut-resistant | 0.6217 | 0.3834 | 0.0000 |

The predefined exploratory gate failed. It requires positive macro-F1 changes on both external datasets and an ArtiFact loss of at most 0.03. Because those external sets informed the intervention, a gate pass still requires an untouched confirmation dataset.
